In [ ]:
import pandas as pd
import requests
import time
import random
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv
import json
import logging
from urllib.parse import quote
import openpyxl
from openpyxl.styles import Font, Alignment
import warnings
warnings.filterwarnings('ignore')

class NaverImageAPISearcher:
    def __init__(self, search_year=2024, search_month=6, max_total_images=8000):
        load_dotenv()
        
        self.client_id = os.getenv('Client_ID')
        self.client_secret = os.getenv('Client_Secret')
        
        if not self.client_id or not self.client_secret:
            raise ValueError("네이버 API 키가 설정되지 않았습니다.")
        
        self.search_year = search_year
        self.search_month = search_month
        self.max_total_images = max_total_images
        self.current_total_count = 0
        
        self.api_url = "https://openapi.naver.com/v1/search/image"
        self.headers = {
            'X-Naver-Client-Id': self.client_id,
            'X-Naver-Client-Secret': self.client_secret
        }
        
        self.results = []
        self.collected_urls = set()
        self.menu_popularity = {}
        
        logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
        self.logger = logging.getLogger(__name__)
        
        self.request_delay = 0.1
        self.daily_request_count = 0
        self.max_daily_requests = 20000
        
        print(f"네이버 이미지 API 검색기 초기화 완료")
        print(f"검색 기간: {search_year}년 {search_month}월")
        print(f"목표 수집 이미지: {max_total_images}개")
    
    def load_menu_data(self, csv_file_path):
        try:
            df = pd.read_csv(csv_file_path, encoding='utf-8')
            self.logger.info(f"CSV 파일 로드 완료: {len(df)}개 행")
            
            menu_items = []
            for _, row in df.iterrows():
                detail_menus = [menu.strip() for menu in str(row['상세메뉴']).split(',') if menu.strip()]
                for detail_menu in detail_menus:
                    menu_items.append({
                        '대분류': row['대분류'],
                        '중분류': row['중분류'],
                        '소분류': row['소분류'],
                        '상세메뉴': detail_menu,
                        '시각적특징': row['시각적특징']
                    })
            
            self.logger.info(f"총 {len(menu_items)}개의 개별 메뉴 항목 생성")
            return menu_items
            
        except Exception as e:
            self.logger.error(f"CSV 파일 로드 실패: {e}")
            raise
    
    def search_images_api(self, query, start=1, display=100, sort='date'):
        try:
            if self.daily_request_count >= self.max_daily_requests:
                self.logger.warning("일일 API 요청 한도에 도달했습니다.")
                return None
            
            params = {
                'query': query,
                'start': start,
                'display': min(display, 100),
                'sort': sort,
                'filter': 'all'
            }
            
            response = requests.get(self.api_url, headers=self.headers, params=params, timeout=10)
            self.daily_request_count += 1
            
            if response.status_code == 200:
                result = response.json()
                return result
            elif response.status_code == 429:
                print(f"API 제한 도달, 10초 대기...")
                time.sleep(10)
                return None
            else:
                self.logger.error(f"API 요청 실패: {response.status_code}")
                return None
                
        except Exception as e:
            self.logger.error(f"API 요청 중 오류: {e}")
            return None
        finally:
            time.sleep(self.request_delay)
    
    def is_valid_date(self, pub_date):
        if not pub_date:
            return False
        
        try:
            date_parts = pub_date.split()
            if len(date_parts) >= 4:
                year = int(date_parts[3])
                month_name = date_parts[2]
                
                month_map = {
                    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
                    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
                    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
                }
                
                month_num = month_map.get(month_name, 0)
                return year == self.search_year and month_num == self.search_month
            
        except Exception:
            pass
        
        return False
    
    def calculate_menu_popularity_by_period(self, menu_items):
        print(f"메뉴별 {self.search_year}년 {self.search_month}월 인기도 측정 중...")
        popularity_scores = {}
        
        for i, menu_item in enumerate(menu_items):
            menu_name = menu_item['상세메뉴']
            print(f"[{i+1}/{len(menu_items)}] '{menu_name}' 해당 시점 인기도 측정")
            
            period_count = 0
            start = 1
            max_check_pages = 5  # 최대 500개까지 확인
            
            for page in range(max_check_pages):
                result = self.search_images_api(menu_name, start=start, display=100, sort='date')
                
                if not result or 'items' not in result:
                    break
                
                images = result['items']
                if not images:
                    break
                
                # 해당 시점의 이미지만 카운트
                valid_count = 0
                for img in images:
                    if self.is_valid_date(img.get('pubDate', '')):
                        valid_count += 1
                
                period_count += valid_count
                print(f"    페이지 {page+1}: {valid_count}개 (누적: {period_count}개)")
                
                # 날짜순 정렬에서 해당 시점을 벗어나기 시작하면 중단
                if valid_count == 0 and page > 0:
                    break
                
                start += len(images)
                
                if len(images) < 100:
                    break
                
                time.sleep(random.uniform(0.1, 0.2))
            
            popularity_scores[menu_name] = period_count
            print(f"  -> {self.search_year}년 {self.search_month}월 총 {period_count}개")
            
            time.sleep(random.uniform(0.2, 0.4))
        
        self.menu_popularity = popularity_scores
        
        # 인기도 순으로 정렬
        sorted_popularity = sorted(popularity_scores.items(), key=lambda x: x[1], reverse=True)
        
        print(f"\n{self.search_year}년 {self.search_month}월 메뉴 인기도 TOP 15")
        for i, (menu, count) in enumerate(sorted_popularity[:15]):
            print(f"{i+1:2d}. {menu:<15}: {count:4d}개")
        
        # 인기도 통계
        total_images = sum(popularity_scores.values())
        non_zero_menus = len([count for count in popularity_scores.values() if count > 0])
        print(f"\n인기도 측정 결과:")
        print(f"총 해당 시점 이미지: {total_images}개")
        print(f"이미지가 있는 메뉴: {non_zero_menus}개 / {len(menu_items)}개")
        
        return popularity_scores
    
    def calculate_menu_quotas(self, menu_items, popularity_scores):
        total_popularity = sum(popularity_scores.values())
        
        if total_popularity == 0:
            print("해당 시점에 이미지가 없어 균등 분배합니다.")
            quota_per_menu = self.max_total_images // len(menu_items)
            return {item['상세메뉴']: quota_per_menu for item in menu_items}
        
        quotas = {}
        
        for menu_item in menu_items:
            menu_name = menu_item['상세메뉴']
            popularity = popularity_scores.get(menu_name, 0)
            
            if total_popularity > 0:
                ratio = popularity / total_popularity
                quota = int(self.max_total_images * ratio)
            else:
                quota = 0
            
            quotas[menu_name] = quota
        
        print(f"\n메뉴별 수집 할당량 TOP 15 (총 {self.max_total_images}개 배분)")
        sorted_quotas = sorted(quotas.items(), key=lambda x: x[1], reverse=True)
        allocated_total = 0
        for i, (menu, quota) in enumerate(sorted_quotas[:15]):
            print(f"{i+1:2d}. {menu:<15}: {quota:4d}개")
            allocated_total += quota
        
        print(f"\n할당량 합계: {allocated_total}개")
        if allocated_total < self.max_total_images:
            remaining = self.max_total_images - allocated_total
            print(f"미할당량: {remaining}개 (인기 메뉴에 추가 배분)")
        
        return quotas
    
    def search_menu_comprehensively(self, menu_item, target_quota):
        menu_name = menu_item['상세메뉴']
        collected_images = []
        
        if target_quota == 0:
            return []
        
        search_keywords = [
            menu_name,
            f"{menu_name} 음식",
            f"{menu_name} 요리",
            f"음식 {menu_name}",
            f"한국음식 {menu_name}",
            f"{menu_name} 이미지"
        ]
        
        print(f"  검색 키워드: {len(search_keywords)}개")
        
        for keyword in search_keywords:
            if len(collected_images) >= target_quota:
                break
            
            for sort_method in ['date', 'sim']:
                if len(collected_images) >= target_quota:
                    break
                
                start = 1
                
                while len(collected_images) < target_quota:
                    result = self.search_images_api(keyword, start=start, display=100, sort=sort_method)
                    
                    if not result or 'items' not in result:
                        break
                    
                    images = result['items']
                    if not images:
                        break
                    
                    valid_images = []
                    for img in images:
                        img_url = img.get('link', '')
                        pub_date = img.get('pubDate', '')
                        
                        if (img_url and 
                            img_url not in self.collected_urls and 
                            self.is_valid_date(pub_date)):
                            
                            valid_images.append(img)
                            self.collected_urls.add(img_url)
                    
                    if valid_images:
                        collected_images.extend(valid_images)
                        print(f"    '{keyword}' ({sort_method}): +{len(valid_images)}개")
                    
                    start += len(images)
                    
                    if len(images) < 100:
                        break
                    
                    # 날짜순에서 해당 시점을 벗어나면 중단
                    if sort_method == 'date' and not any(self.is_valid_date(img.get('pubDate', '')) for img in images):
                        break
        
        final_images = collected_images[:target_quota]
        
        processed_images = []
        for img in final_images:
            processed_img = self.process_image_data(img, menu_item, f"포괄검색:{menu_name}")
            processed_images.append(processed_img)
        
        return processed_images
    
    def process_image_data(self, api_image, menu_item, search_keyword):
        return {
            'image_url': api_image.get('link', ''),
            'thumbnail_url': api_image.get('thumbnail', ''),
            'title': api_image.get('title', '').replace('<b>', '').replace('</b>', ''),
            'size_height': api_image.get('sizeheight', ''),
            'size_width': api_image.get('sizewidth', ''),
            'pub_date': api_image.get('pubDate', ''),
            '대분류': menu_item['대분류'],
            '중분류': menu_item['중분류'],
            '소분류': menu_item['소분류'],
            '상세메뉴': menu_item['상세메뉴'],
            '시각적특징': menu_item['시각적특징'],
            '업로드시기': f"{self.search_year}-{self.search_month:02d}",
            '검색키워드': search_keyword,
            'collected_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
    
    def process_all_menus(self, csv_file_path):
        menu_items = self.load_menu_data(csv_file_path)
        
        # 해당 시점 인기도 측정
        popularity_scores = self.calculate_menu_popularity_by_period(menu_items)
        menu_quotas = self.calculate_menu_quotas(menu_items, popularity_scores)
        
        print(f"\n메뉴별 이미지 수집 시작")
        processed_count = 0
        
        # 할당량이 있는 메뉴만 처리
        valid_menus = [(item, menu_quotas.get(item['상세메뉴'], 0)) 
                       for item in menu_items if menu_quotas.get(item['상세메뉴'], 0) > 0]
        
        print(f"수집 대상 메뉴: {len(valid_menus)}개 (할당량 0인 메뉴 제외)")
        
        for i, (menu_item, target_quota) in enumerate(valid_menus):
            if self.current_total_count >= self.max_total_images:
                print(f"\n목표 수집량({self.max_total_images}개)에 도달하여 중단합니다.")
                break
            
            if self.daily_request_count >= self.max_daily_requests:
                print(f"\nAPI 일일 한도에 도달하여 중단합니다.")
                break
            
            menu_name = menu_item['상세메뉴']
            
            print(f"\n[{i+1}/{len(valid_menus)}] '{menu_name}' 수집 (할당량: {target_quota}개)")
            
            images = self.search_menu_comprehensively(menu_item, target_quota)
            
            if images:
                self.results.extend(images)
                self.current_total_count += len(images)
                
                success_rate = (len(images) / target_quota * 100) if target_quota > 0 else 0
                print(f"  -> {len(images)}개 수집 완료 (달성률: {success_rate:.1f}%)")
                print(f"  -> 총 누적: {self.current_total_count}개")
            else:
                print(f"  -> 수집된 이미지 없음")
            
            processed_count += 1
            
            if processed_count % 10 == 0:
                progress = (processed_count / len(valid_menus)) * 100
                overall_success = (self.current_total_count / self.max_total_images * 100)
                print(f"\n진행률: {progress:.1f}% ({processed_count}/{len(valid_menus)})")
                print(f"전체 달성률: {overall_success:.1f}% ({self.current_total_count}/{self.max_total_images})")
                print(f"API 요청: {self.daily_request_count}회")
            
            time.sleep(random.uniform(0.5, 1.0))
        
        print(f"\n전체 처리 완료!")
        print(f"최종 수집 이미지: {self.current_total_count}개")
        print(f"목표 달성률: {(self.current_total_count/self.max_total_images)*100:.1f}%")
        print(f"처리된 메뉴: {processed_count}개")
        print(f"API 요청 횟수: {self.daily_request_count}회")
    
    def save_to_excel(self, output_file="naver_image_period_based.xlsx"):
        if not self.results:
            print("저장할 결과가 없습니다.")
            return
        
        try:
            df = pd.DataFrame(self.results)
            
            columns_order = [
                '상세메뉴', '대분류', '중분류', '소분류', '시각적특징',
                'image_url', 'thumbnail_url', 'title', 
                'size_width', 'size_height', 'pub_date',
                '업로드시기', '검색키워드', 'collected_at'
            ]
            
            available_columns = [col for col in columns_order if col in df.columns]
            df = df[available_columns]
            
            original_count = len(df)
            df = df.drop_duplicates(subset=['image_url'], keep='first')
            removed_count = original_count - len(df)
            
            if removed_count > 0:
                print(f"중복 제거: {removed_count}개 (최종: {len(df)}개)")
            
            with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
                df.to_excel(writer, sheet_name='검색결과', index=False)
                
                worksheet = writer.sheets['검색결과']
                
                header_font = Font(bold=True)
                header_alignment = Alignment(horizontal='center')
                
                for cell in worksheet[1]:
                    cell.font = header_font
                    cell.alignment = header_alignment
                
                column_widths = {
                    'image_url': 50,
                    'thumbnail_url': 50,
                    'title': 35,
                    '상세메뉴': 15,
                    '시각적특징': 25,
                    '검색키워드': 25
                }
                
                for column in worksheet.columns:
                    column_letter = column[0].column_letter
                    column_name = column[0].value
                    
                    if column_name in column_widths:
                        worksheet.column_dimensions[column_letter].width = column_widths[column_name]
                    else:
                        max_length = 0
                        for cell in column:
                            try:
                                if len(str(cell.value)) > max_length:
                                    max_length = len(str(cell.value))
                            except:
                                pass
                        adjusted_width = min(max_length + 2, 25)
                        worksheet.column_dimensions[column_letter].width = adjusted_width
            
            self.save_statistics(output_file, df)
            
            print(f"\n결과 저장 완료: {output_file}")
            print(f"최종 저장: {len(df)}개 이미지 정보")
            
        except Exception as e:
            self.logger.error(f"엑셀 저장 실패: {e}")
            
            try:
                df.to_csv(output_file.replace('.xlsx', '.csv'), encoding='utf-8-sig', index=False)
                print(f"백업 CSV 파일로 저장: {output_file.replace('.xlsx', '.csv')}")
            except Exception as csv_error:
                self.logger.error(f"CSV 백업 저장도 실패: {csv_error}")
    
    def save_statistics(self, excel_file, df):
        try:
            menu_stats = df.groupby('상세메뉴').size().reset_index(name='실제수집량')
            
            # 인기도와 할당량 정보 추가
            quota_comparison = []
            for _, row in menu_stats.iterrows():
                menu_name = row['상세메뉴']
                actual = row['실제수집량']
                popularity = self.menu_popularity.get(menu_name, 0)
                
                # 할당량 재계산
                total_popularity = sum(self.menu_popularity.values())
                if total_popularity > 0:
                    quota = int(self.max_total_images * (popularity / total_popularity))
                else:
                    quota = 0
                
                quota_comparison.append({
                    '메뉴명': menu_name,
                    '시점인기도': popularity,
                    '할당량': quota,
                    '실제수집': actual,
                    '달성률': f"{(actual/quota*100):.1f}%" if quota > 0 else "0%"
                })
            
            quota_df = pd.DataFrame(quota_comparison)
            quota_df = quota_df.sort_values('실제수집', ascending=False)
            
            category_stats = df.groupby(['대분류', '중분류']).size().reset_index(name='이미지수')
            category_stats = category_stats.sort_values('이미지수', ascending=False)
            
            # 시점별 인기도 통계
            popularity_df = pd.DataFrame(list(self.menu_popularity.items()), 
                                       columns=['메뉴명', '시점인기도'])
            popularity_df = popularity_df.sort_values('시점인기도', ascending=False)
            
            df['수집일'] = df['pub_date'].apply(lambda x: x.split()[1:4] if x else ['', '', ''])
            df['수집일str'] = df['수집일'].apply(lambda x: f"{x[2]}-{x[1]}-{x[0]}" if len(x) == 3 else "날짜없음")
            date_stats = df.groupby('수집일str').size().reset_index(name='이미지수')
            date_stats = date_stats.sort_values('이미지수', ascending=False)
            
            with pd.ExcelWriter(excel_file, mode='a', engine='openpyxl') as writer:
                quota_df.to_excel(writer, sheet_name='할당량대비실적', index=False)
                popularity_df.to_excel(writer, sheet_name='시점인기도순위', index=False)
                menu_stats.to_excel(writer, sheet_name='메뉴별수집량', index=False)
                category_stats.to_excel(writer, sheet_name='분류별통계', index=False)
                date_stats.to_excel(writer, sheet_name='날짜별분포', index=False)
                
                total_popularity = sum(self.menu_popularity.values())
                success_rate = (len(df) / self.max_total_images * 100) if self.max_total_images > 0 else 0
                
                summary_data = {
                    '항목': [
                        '최종 수집 이미지',
                        '목표 달성률',
                        '시점 총 인기도',
                        '인기도 기반 할당',
                        '처리된 메뉴 수',
                        '평균 메뉴당 수집',
                        '검색 기간 (정확)',
                        '수집 완료 시각',
                        'API 총 요청',
                        '날짜 정보 정확성'
                    ],
                    '값': [
                        f"{len(df)}개",
                        f"{success_rate:.1f}%",
                        f"{total_popularity}개",
                        "비례 할당",
                        f"{df['상세메뉴'].nunique()}개",
                        f"{len(df) / df['상세메뉴'].nunique():.1f}개" if df['상세메뉴'].nunique() > 0 else "0개",
                        f"{self.search_year}년 {self.search_month}월만",
                        datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                        f"{self.daily_request_count}회",
                        f"{(len(df[df['pub_date'] != '']) / len(df) * 100):.1f}%" if len(df) > 0 else "0%"
                    ]
                }
                
                summary_df = pd.DataFrame(summary_data)
                summary_df.to_excel(writer, sheet_name='수집요약', index=False)
            
        except Exception as e:
            self.logger.error(f"통계 저장 실패: {e}")


def main():
    CSV_FILE_PATH = "식당대12중53소132상세메뉴379분류.csv"
    SEARCH_YEAR = 2024
    SEARCH_MONTH = 6
    MAX_IMAGES = 8000
    OUTPUT_FILE = f"naver_image_period_based_{SEARCH_YEAR}{SEARCH_MONTH:02d}.xlsx"
    
    searcher = None
    
    try:
        print("네이버 이미지 시점별 인기도 기반 수집 시스템")
        
        searcher = NaverImageAPISearcher(
            search_year=SEARCH_YEAR,
            search_month=SEARCH_MONTH,
            max_total_images=MAX_IMAGES
        )
        
        if not os.path.exists(CSV_FILE_PATH):
            print(f"오류: CSV 파일을 찾을 수 없습니다 - {CSV_FILE_PATH}")
            return
        
        if not os.path.exists('.env'):
            print("\n.env 파일이 필요합니다:")
            print("NAVER_CLIENT_ID=your_client_id")
            print("NAVER_CLIENT_SECRET=your_client_secret")
            return
        
        print(f"\n{SEARCH_YEAR}년 {SEARCH_MONTH}월 시점별 인기도 기반 수집 시작...")
        
        searcher.process_all_menus(CSV_FILE_PATH)
        searcher.save_to_excel(OUTPUT_FILE)
        
        print("\n작업 완료!")
        
    except ValueError as ve:
        print(f"\n설정 오류: {ve}")
        
    except KeyboardInterrupt:
        print("\n사용자에 의해 중단되었습니다.")
        if searcher and searcher.results:
            print("현재까지의 결과를 저장합니다...")
            searcher.save_to_excel(f"partial_{OUTPUT_FILE}")
        
    except Exception as e:
        print(f"\n오류 발생: {e}")
        import traceback
        traceback.print_exc()
        
    finally:
        if searcher:
            print("\n시스템 종료")


if __name__ == "__main__":
    main()

2025-07-15 00:41:56,062 - INFO - CSV 파일 로드 완료: 138개 행
2025-07-15 00:41:56,094 - INFO - 총 381개의 개별 메뉴 항목 생성


네이버 이미지 시점별 인기도 기반 수집 시스템
네이버 이미지 API 검색기 초기화 완료
검색 기간: 2024년 6월
목표 수집 이미지: 8000개

2024년 6월 시점별 인기도 기반 수집 시작...
메뉴별 2024년 6월 인기도 측정 중...
[1/381] '제육볶음' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[2/381] '매운제육볶음' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[3/381] '두부제육볶음' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[4/381] '된장찌개' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[5/381] '김치찌개' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[6/381] '청국장찌개' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[7/381] '콩나물무침' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[8/381] '시금치나물' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[9/381] '도라지무침' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[10/381] '계란말이' 해당

[87/381] '회덮밥' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[88/381] '조개구이' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[89/381] '키조개' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[90/381] '가리비' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[91/381] '파전' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[92/381] '김치전' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[93/381] '해물전' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[94/381] '골뱅이무침' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[95/381] '오징어무침' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[96/381] '멍게무침' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[97/381] '곱창' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (

    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[175/381] '모둠사시미' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[176/381] '연어덮밥' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[177/381] '장어덮밥' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[178/381] '김밥롤' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[179/381] '캘리포니아롤' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[180/381] '필라델피아롤' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[181/381] '돈코츠라멘' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[182/381] '차슈라멘' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[183/381] '미소라멘' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[184/381] '쇼유라멘' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)

2025-07-15 03:06:28,181 - ERROR - API 요청 중 오류: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


  -> 2024년 6월 총 0개
[194/381] '규동' 해당 시점 인기도 측정


2025-07-15 06:10:57,323 - ERROR - API 요청 중 오류: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


  -> 2024년 6월 총 0개
[195/381] '오야코동' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[196/381] '텐동' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[197/381] '가케우동' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[198/381] '텐푸라우동' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[199/381] '카레우동' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[200/381] '자루소바' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[201/381] '온소바' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[202/381] '메밀소바' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[203/381] '돈까스' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[204/381] '치킨가츠' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[205/381] '생선까스' 해당 시점 인기도 측정
    

[282/381] '똠얌꿍' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[283/381] '똠얌갈비' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[284/381] '새콤매운국물' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[285/381] '치킨커리' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[286/381] '양고기커리' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[287/381] '달커리' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[288/381] '난' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[289/381] '파라타' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[290/381] '탄두리치킨' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[291/381] '케밥' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[292/381] '구이' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
   

    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[369/381] '샐러드바' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[370/381] '샐러드뷔페' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[371/381] '호텔뷔페' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[372/381] '브런치뷔페' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[373/381] '비건버거' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[374/381] '두부스테이크' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[375/381] '템페' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[376/381] '비건케이크' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[377/381] '두유아이스크림' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: 0개)
  -> 2024년 6월 총 0개
[378/381] '닭가슴살샐러드' 해당 시점 인기도 측정
    페이지 1: 0개 (누적: 0개)
    페이지 2: 0개 (누적: